# Kafka

In [ ]:
from confluent_kafka import Producer

conf = {
    "bootstrap.servers": "localhost:9092"
}
producer = Producer(conf)
producer.produce("my_topic", value="hello world")
producer.flush()


In [ ]:
from confluent_kafka import Consumer

# Config
conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'test-group',
    'auto.offset.reset': 'earliest'  # start from beginning if no offset
}

consumer = Consumer(conf)
consumer.subscribe(['weather-raw'])

print("Consuming...")
while True:
    msg = consumer.poll(1.0)
    if msg is None:
        continue
    if msg.error():
        print("Error:", msg.error())
        continue
    value = msg.value().decode('utf-8')
    print(f"type: {type(value)} - {value}")

consumer.close()


# spark stream

In [23]:
%%sh
wget -O ./jars/spark-sql-kafka-0-10_2.13-3.2.4.jar https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.13/3.2.4/spark-sql-kafka-0-10_2.13-3.2.4.jar

--2025-05-03 13:20:09--  https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.13/3.2.4/spark-sql-kafka-0-10_2.13-3.2.4.jar
Resolving repo1.maven.org (repo1.maven.org)... 146.75.40.209, 151.101.196.209, 2a04:4e42:a::209, ...
Connecting to repo1.maven.org (repo1.maven.org)|146.75.40.209|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 426685 (417K) [application/java-archive]
Saving to: ‘./jars/spark-sql-kafka-0-10_2.13-3.2.4.jar’

     0K .......... .......... .......... .......... .......... 11%  193K 2s
    50K .......... .......... .......... .......... .......... 23%  177K 2s
   100K .......... .......... .......... .......... .......... 35% 13.2M 1s
   150K .......... .......... .......... .......... .......... 47% 32.9M 1s
   200K .......... .......... .......... .......... .......... 59%  196K 1s
   250K .......... .......... .......... .......... .......... 71%  613K 0s
   300K .......... .......... .......... .......... .......... 83%  

In [6]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf
from contextlib import contextmanager
from pyspark.sql.types import StructType, StringType, IntegerType, TimestampType, StructField
from pyspark.sql.functions import col, regexp_replace, trim, regexp_extract, to_timestamp, from_json
from pyspark.sql.types import DoubleType
import json

from pyspark.sql import SparkSession, functions as F, types as T
import logging


@contextmanager
def SparkIO(conf: SparkConf = SparkConf()):
            # gcs: bool = False):
    app_name = conf.get("spark.app.name")
    master = conf.get("spark.master")

    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    logging.info(f'Create SparkSession app {app_name} with {master} mode')
    try:
        yield spark
    except Exception as e:
        logging.error(e)
        raise 
    finally:
        logging.info(f'Stop SparkSession app {app_name}')
        spark.stop()

In [2]:
packages = [
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.4"
]

conf = (SparkConf().setAppName("Weather-Kafka-Druid")
    .set("spark.executor.memory", "2g")
    .set("spark.jars.packages", ",".join(packages))
    .setMaster("local[*]")
    )

with SparkIO(conf=conf) as spark:
    
    # Read from Kafka
    df = spark.readStream\
        .format("kafka")\
        .option("kafka.bootstrap.servers", "kafka-broker-1:9094")\
        .option("subscribe", "weather-raw")\
        .load()

    df = df.selectExpr("CAST(value AS STRING)")
    schema = StructType() \
        .add("current_time", StringType()) \
        .add("status", StringType()) \
        .add("temp_c", StringType()) \
        .add("realfeel®", StringType()) \
        .add("wind", StringType()) \
        .add("wind gusts", StringType()) \
        .add("humidity", StringType()) \
        .add("indoor humidity", StringType()) \
        .add("dew point", StringType()) \
        .add("pressure", StringType()) \
        .add("cloud cover", StringType()) \
        .add("visibility", StringType()) \
        .add("cloud ceiling", StringType()) \
        .add("timestamp", StringType())
    
    parsed_df = df.select(from_json(col("value"), schema).alias("data")).select("data.*")

    # Clean and standardize
    cleaned_df = parsed_df \
        .withColumn("temp_c", regexp_extract(col("temp_c"), r"([\d.]+)", 1).cast(DoubleType())) \
        .withColumn("realfeel", regexp_extract(col("realfeel®"), r"([\d.]+)", 1).cast(DoubleType())) \
        .withColumn("wind_kmh", regexp_extract(col("wind"), r"(\d+)", 1).cast(DoubleType())) \
        .withColumn("wind_gusts_kmh", regexp_extract(col("wind gusts"), r"(\d+)", 1).cast(DoubleType())) \
        .withColumn("humidity", regexp_replace(col("humidity"), "%", "").cast(DoubleType())) \
        .withColumn("indoor_humidity", regexp_extract(col("indoor humidity"), r"(\d+)", 1).cast(DoubleType())) \
        .withColumn("dew_point_c", regexp_extract(col("dew point"), r"([\d.]+)", 1).cast(DoubleType())) \
        .withColumn("pressure_mb", regexp_extract(col("pressure"), r"(\d+)", 1).cast(DoubleType())) \
        .withColumn("cloud_cover", regexp_replace(col("cloud cover"), "%", "").cast(DoubleType())) \
        .withColumn("visibility_km", regexp_extract(col("visibility"), r"([\d.]+)", 1).cast(DoubleType())) \
        .withColumn("cloud_ceiling_m", regexp_extract(col("cloud ceiling"), r"(\d+)", 1).cast(DoubleType())) \
        .withColumn("timestamp", to_timestamp("timestamp"))  # Druid likes proper timestamps

    # Optional: drop original columns with odd names or keep selected ones
    final_df = cleaned_df.select(
        "timestamp",
        "temp_c",
        "realfeel",
        "wind_kmh",
        "wind_gusts_kmh",
        "humidity",
        "indoor_humidity",
        "dew_point_c",
        "pressure_mb",
        "cloud_cover",
        "visibility_km",
        "cloud_ceiling_m"
    )

    # Write to console
    query = final_df.writeStream\
        .outputMode("append")\
        .format("console")\
        .option("truncate", False)\
        .start()
    query.awaitTermination()

:: loading settings :: url = jar:file:/opt/conda/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6741e3aa-9945-4f59-82a5-6b7fb82f0457;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.2.4 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.2.4 in central
	found org.apache.kafka#kafka-clients;2.8.1 in central
	found org.lz4#lz4-java;1.7.1 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.30 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.1 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.1 in central
	found org.apache.htrace#htrace-core4;4.1.0-incubating in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central

-------------------------------------------
Batch: 0
-------------------------------------------
+---------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+
|timestamp|temp_c|realfeel|wind_kmh|wind_gusts_kmh|humidity|indoor_humidity|dew_point_c|pressure_mb|cloud_cover|visibility_km|cloud_ceiling_m|
+---------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+
+---------+------+--------+--------+--------------+--------+---------------+-----------+-----------+-----------+-------------+---------------+



ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
RuntimeError: reentrant call inside <_io.BufferedReader name=62>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/conda/lib/python3.9/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.9/soc

Exception: 

## Traffic processing

In [9]:
KAFKA_SERVER = "kafka-broker-1:9094"
KAFKA_TOPIC = "weather-raw"

with SparkIO(conf=conf) as spark:
    object_schema = T.StructType([
        T.StructField("class_id", T.IntegerType()),
        T.StructField("class_object", T.StringType()),
        T.StructField("classname", T.StringType()),
        T.StructField("confidence", T.DoubleType()),
        T.StructField("coordinates", T.ArrayType(T.DoubleType()))
    ])

    traffic_schema = T.StructType([
        T.StructField("cam_id", T.StringType()),
        T.StructField("img", T.StringType()),
        T.StructField("district", T.StringType()),
        T.StructField("timestamp", T.StringType()),
        T.StructField("image_shape", T.ArrayType(T.IntegerType())),
        T.StructField("total", T.IntegerType()),
        T.StructField("objects", T.ArrayType(object_schema)),
    ])


    weather_schema = StructType([
        StructField("cloud ceiling", StringType(), True),
        StructField("cloud cover", StringType(), True),
        StructField("current_time", StringType(), True),
        StructField("dew point", StringType(), True),
        StructField("humidity", StringType(), True),
        StructField("indoor humidity", StringType(), True),
        StructField("max uv index", StringType(), True),
        StructField("pressure", StringType(), True),
        StructField("realfeel shade™", StringType(), True),
        StructField("realfeel®", StringType(), True),
        StructField("status", StringType(), True),
        StructField("temp_c", StringType(), True),
        StructField("timestamp", StringType(), True),
        StructField("visibility", StringType(), True),
        StructField("wind", StringType(), True),
        StructField("wind gusts", StringType(), True),
        StructField("day", IntegerType(), True),
        StructField("hour", IntegerType(), True),
        StructField("district", StringType())
    ])

    def transform_traffic(df):

        # Cast type
        trans_df = (df
                .withColumn("cam_id", F.trim(F.col("cam_id")).alias("cam_id"))
                .withColumn("ts", F.col("timestamp").cast("timestamp"))
                .withColumn("image_h", F.element_at("image_shape", 1).cast("int"))
                .withColumn("image_w", F.element_at("image_shape", 2).cast("int"))
        )
        
        # Explode dict object
        trans_df = trans_df.withColumn("obj", F.explode_outer("objects"))
        trans_df = (trans_df
                .withColumn("object_id", F.col("obj.class_id").cast("int"))
                .withColumn("object_name", F.col("obj.class_object").cast("string"))
                .withColumn("object_confidence",
                            F.when(F.col("obj.confidence").between(0.0, 1.0),
                                F.col("obj.confidence")))
                .withColumn("object_bbox",
                            F.when(F.size("obj.coordinates") == 4,
                                F.col("obj.coordinates").cast("array<double>")))
        )

        trans_df = trans_df.filter(
                (F.col("ts").isNotNull())
            )

        trans_df = trans_df \
                    .withColumn("hour_of_day", F.hour(F.col("ts"))) \
                    .withColumn("dt", F.to_date("ts"))

        cols = ["cam_id", "district", "image_w", "image_h",
                    "object_name", "object_confidence",
                    "object_bbox", "ts", "hour_of_day", 'dt']

        return trans_df.select(*cols)

    def transform_weather(df):

        def extract_number(col):
            return F.regexp_extract(col, r"(\d+)", 1).cast("int")

        def extract_uv_status(col):
            return F.lower(F.trim(F.regexp_extract(col, r"\d+\s*(\w+)", 1)))

        def extract_wind_direction(col):
            return F.regexp_extract(col, r"^([A-Z]+)", 1)

        trans_df = (
            df
            .withColumn("ts", F.col("timestamp").cast("timestamp"))
            .withColumn("cloud_ceiling_m", extract_number("cloud ceiling"))
            .withColumn("cloud_cover_pct", extract_number("cloud cover"))
            .withColumn("dew_point_c", extract_number("dew point"))
            .withColumn("humidity_pct", extract_number("humidity"))
            .withColumn("uv_index", extract_number("max uv index"))
            .withColumn("uv_index_status", extract_uv_status("max uv index"))
            .withColumn("pressure_mb", extract_number("pressure"))
            .withColumn("realfeel_c", extract_number("realfeel®"))
            .withColumn("realfeel_shade_c", extract_number("realfeel shade™"))
            .withColumn("temp_c", extract_number("temp_c"))
            .withColumn("visibility_km", extract_number("visibility"))
            .withColumn("wind_kmph", extract_number("wind"))
            .withColumn("wind_direction", extract_wind_direction("wind"))
            .withColumn("wind_gust_kmph", extract_number("wind gusts"))
            .withColumn("status", F.lower(F.col("status")))
            .withColumn("dt", F.to_date("ts"))
            .withColumn("hour_of_day", F.hour("ts"))
        )

        # Select relevant fields
        cols = [
            "ts", "district", "cloud_ceiling_m", "cloud_cover_pct", "dew_point_c",
            "humidity_pct", "uv_index", "uv_index_status", "pressure_mb", "realfeel_c",
            "realfeel_shade_c", "temp_c", "visibility_km", "status",
            "wind_kmph", "wind_direction", "wind_gust_kmph", "dt", "hour_of_day"
        ]

        return trans_df.select(*cols)

    def read_kafka_stream(topic, schema):
        return (spark.readStream
                .format("kafka")
                .option("kafka.bootstrap.servers", KAFKA_SERVER)
                .option("subscribe", topic)
                .option("startingOffsets", "latest")
                .load()
                .selectExpr("CAST(value AS STRING) as json")
                .select(F.from_json("json", schema).alias("data"))
                .select("data.*"))

    weather_raw = read_kafka_stream("weather-raw", weather_schema)
    # traffic_raw = read_kafka_stream("traffic-object-raw", traffic_schema)

    # traffic_clean = transform_traffic(traffic_raw)
    weather_clean = transform_weather(weather_raw)

    query = weather_clean.writeStream\
    .outputMode("append")\
    .format("console")\
    .option("truncate", False)\
    .start()

    query.awaitTermination()

25/05/31 15:58:28 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-060ec696-ff03-4032-9e0b-7c4831ce1572. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/05/31 15:58:28 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+---+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+---+-----------+
|ts |district|cloud_ceiling_m|cloud_cover_pct|dew_point_c|humidity_pct|uv_index|uv_index_status|pressure_mb|realfeel_c|realfeel_shade_c|temp_c|visibility_km|status|wind_kmph|wind_direction|wind_gust_kmph|dt |hour_of_day|
+---+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+---+-----------+
+---+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+---+-----------+



25/05/31 16:03:48 ERROR WriteToDataSourceV2Exec: Data source write support org.apache.spark.sql.execution.streaming.sources.MicroBatchWrite@2595e90e is aborting.
25/05/31 16:03:48 ERROR WriteToDataSourceV2Exec: Data source write support org.apache.spark.sql.execution.streaming.sources.MicroBatchWrite@2595e90e aborted.
25/05/31 16:03:49 ERROR MicroBatchExecution: Query [id = 29bae0f2-cbd0-409f-ba98-2e3718a7f64f, runId = 7f21bd13-d2ba-489d-a668-0cbf9ec5caf5] terminated with error
org.apache.spark.SparkException: Writing job aborted
	at org.apache.spark.sql.errors.QueryExecutionErrors$.writingJobAbortedError(QueryExecutionErrors.scala:613)
	at org.apache.spark.sql.execution.datasources.v2.V2TableWriteExec.writeWithV2(WriteToDataSourceV2Exec.scala:386)
	at org.apache.spark.sql.execution.datasources.v2.V2TableWriteExec.writeWithV2$(WriteToDataSourceV2Exec.scala:330)
	at org.apache.spark.sql.execution.datasources.v2.WriteToDataSourceV2Exec.writeWithV2(WriteToDataSourceV2Exec.scala:279)
	at o

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------------+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+----------+-----------+
|ts                 |district|cloud_ceiling_m|cloud_cover_pct|dew_point_c|humidity_pct|uv_index|uv_index_status|pressure_mb|realfeel_c|realfeel_shade_c|temp_c|visibility_km|status|wind_kmph|wind_direction|wind_gust_kmph|dt        |hour_of_day|
+-------------------+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+----------+-----------+
|2025-05-31 23:03:44|Quận 1  |12200          |91             |25         |88          |null    |null           |1011       |33        |null            |27    |16          

-------------------------------------------
Batch: 2
-------------------------------------------
+-------------------+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+----------+-----------+
|ts                 |district|cloud_ceiling_m|cloud_cover_pct|dew_point_c|humidity_pct|uv_index|uv_index_status|pressure_mb|realfeel_c|realfeel_shade_c|temp_c|visibility_km|status|wind_kmph|wind_direction|wind_gust_kmph|dt        |hour_of_day|
+-------------------+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+----------+-----------+
|2025-05-31 23:04:18|Quận 5  |12200          |91             |25         |88          |null    |null           |1011       |33        |null            |27    |16          

-------------------------------------------
Batch: 4
-------------------------------------------
+-------------------+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+----------+-----------+
|ts                 |district|cloud_ceiling_m|cloud_cover_pct|dew_point_c|humidity_pct|uv_index|uv_index_status|pressure_mb|realfeel_c|realfeel_shade_c|temp_c|visibility_km|status|wind_kmph|wind_direction|wind_gust_kmph|dt        |hour_of_day|
+-------------------+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+----------+-----------+
|2025-05-31 23:05:12|Quận 3  |12200          |91             |25         |88          |null    |null           |1011       |33        |null            |27    |16          

-------------------------------------------
Batch: 6
-------------------------------------------
+-------------------+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+----------+-----------+
|ts                 |district|cloud_ceiling_m|cloud_cover_pct|dew_point_c|humidity_pct|uv_index|uv_index_status|pressure_mb|realfeel_c|realfeel_shade_c|temp_c|visibility_km|status|wind_kmph|wind_direction|wind_gust_kmph|dt        |hour_of_day|
+-------------------+--------+---------------+---------------+-----------+------------+--------+---------------+-----------+----------+----------------+------+-------------+------+---------+--------------+--------------+----------+-----------+
|2025-05-31 23:06:11|Quận 5  |12200          |91             |25         |88          |null    |null           |1011       |33        |null            |27    |16          

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
RuntimeError: reentrant call inside <_io.BufferedReader name=62>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/conda/lib/python3.9/site-packages/py4j/clientserver.py", line 539, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.9/soc

Py4JError: An error occurred while calling o655.awaitTermination

25/05/31 16:08:57 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-a5434744-e174-42fd-ad61-2497706c5456-907134723-driver-0-2, groupId=spark-kafka-source-a5434744-e174-42fd-ad61-2497706c5456-907134723-driver-0] Connection to node 0 (kafka-broker-1/172.22.0.2:9094) could not be established. Broker may not be available.
25/05/31 16:08:58 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-a5434744-e174-42fd-ad61-2497706c5456-907134723-driver-0-2, groupId=spark-kafka-source-a5434744-e174-42fd-ad61-2497706c5456-907134723-driver-0] Connection to node 0 (kafka-broker-1/172.22.0.2:9094) could not be established. Broker may not be available.
25/05/31 16:08:58 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-a5434744-e174-42fd-ad61-2497706c5456-907134723-driver-0-2, groupId=spark-kafka-source-a5434744-e174-42fd-ad61-2497706c5456-907134723-driver-0] Connection to node 0 (kafka-broker-1/172.22.0.2:9094) could not be established. Broker may not be 

In [2]:
with SparkIO("test_app") as spark:
    # Read from Kafka
    df = spark.readStream\
        .format("kafka")\
        .option("kafka.bootstrap.servers", "kafka-broker-1:9094")\
        .option("subscribe", "weather-raw")\
        .load()

    df = df.selectExpr("CAST(value AS STRING)")
    schema = StructType() \
        .add("current_time", StringType()) \
        .add("status", StringType()) \
        .add("temp_c", StringType()) \
        .add("realfeel®", StringType()) \
        .add("wind", StringType()) \
        .add("wind gusts", StringType()) \
        .add("humidity", StringType()) \
        .add("indoor humidity", StringType()) \
        .add("dew point", StringType()) \
        .add("pressure", StringType()) \
        .add("cloud cover", StringType()) \
        .add("visibility", StringType()) \
        .add("cloud ceiling", StringType()) \
        .add("timestamp", StringType())
    
    parsed_df = df.select(from_json(col("value"), schema).alias("data")).select("data.*")

    # Clean and standardize
    cleaned_df = parsed_df \
        .withColumn("temp_c", regexp_extract(col("temp_c"), r"([\d.]+)", 1).cast(DoubleType())) \
        .withColumn("realfeel", regexp_extract(col("realfeel®"), r"([\d.]+)", 1).cast(DoubleType())) \
        .withColumn("wind_kmh", regexp_extract(col("wind"), r"(\d+)", 1).cast(DoubleType())) \
        .withColumn("wind_gusts_kmh", regexp_extract(col("wind gusts"), r"(\d+)", 1).cast(DoubleType())) \
        .withColumn("humidity", regexp_replace(col("humidity"), "%", "").cast(DoubleType())) \
        .withColumn("indoor_humidity", regexp_extract(col("indoor humidity"), r"(\d+)", 1).cast(DoubleType())) \
        .withColumn("dew_point_c", regexp_extract(col("dew point"), r"([\d.]+)", 1).cast(DoubleType())) \
        .withColumn("pressure_mb", regexp_extract(col("pressure"), r"(\d+)", 1).cast(DoubleType())) \
        .withColumn("cloud_cover", regexp_replace(col("cloud cover"), "%", "").cast(DoubleType())) \
        .withColumn("visibility_km", regexp_extract(col("visibility"), r"([\d.]+)", 1).cast(DoubleType())) \
        .withColumn("cloud_ceiling_m", regexp_extract(col("cloud ceiling"), r"(\d+)", 1).cast(DoubleType())) \
        .withColumn("timestamp", to_timestamp("timestamp"))  # Druid likes proper timestamps

    # Optional: drop original columns with odd names or keep selected ones
    final_df = cleaned_df.select(
        "timestamp",
        "temp_c",
        "realfeel",
        "wind_kmh",
        "wind_gusts_kmh",
        "humidity",
        "indoor_humidity",
        "dew_point_c",
        "pressure_mb",
        "cloud_cover",
        "visibility_km",
        "cloud_ceiling_m"
    )

    # Write to console
    query = final_df.writeStream\
        .outputMode("append")\
        .format("console")\
        .option("truncate", False)\
        .start()
    query.awaitTermination()

25/05/31 11:56:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Created SparkSession app test_app
Error in SparkSession app test_app: An error occurred while calling o37.load.
: java.lang.NoClassDefFoundError: scala/$less$colon$less
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.org$apache$spark$sql$kafka010$KafkaSourceProvider$$validateStreamOptions(KafkaSourceProvider.scala:338)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.sourceSchema(KafkaSourceProvider.scala:71)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceSchema(DataSource.scala:236)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo$lzycompute(DataSource.scala:118)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo(DataSource.scala:118)
	at org.apache.spark.sql.execution.streaming.StreamingRelation$.apply(StreamingRelation.scala:34)
	at org.apache.spark.sql.streaming.DataStreamReader.loadInternal(DataStreamReader.scala:167)
	at org.apache.spark.sql.streaming.DataStreamReader.load(DataStreamReader.scala:143)
	at java.base/

Py4JJavaError: An error occurred while calling o37.load.
: java.lang.NoClassDefFoundError: scala/$less$colon$less
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.org$apache$spark$sql$kafka010$KafkaSourceProvider$$validateStreamOptions(KafkaSourceProvider.scala:338)
	at org.apache.spark.sql.kafka010.KafkaSourceProvider.sourceSchema(KafkaSourceProvider.scala:71)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceSchema(DataSource.scala:236)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo$lzycompute(DataSource.scala:118)
	at org.apache.spark.sql.execution.datasources.DataSource.sourceInfo(DataSource.scala:118)
	at org.apache.spark.sql.execution.streaming.StreamingRelation$.apply(StreamingRelation.scala:34)
	at org.apache.spark.sql.streaming.DataStreamReader.loadInternal(DataStreamReader.scala:167)
	at org.apache.spark.sql.streaming.DataStreamReader.load(DataStreamReader.scala:143)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(N

In [ ]:
from pyspark.sql.functions import from_json, col
packages = [
    # "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
    "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    # 'org.apache.kafka:kafka-clients:3.2.4'
]
    .config("spark.sql.repl.eagerEval.enabled", True) \
spark = SparkSession.builder\
         .appName("KafkaSparkStreaming")\
         .config("spark.jars.packages", ",".join(packages))\
         .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

schema = StructType()\
    .add("current_time", StringType(), True)\
    .add("status", StringType(), True)\
    .add("temp_c", StringType(), True)\
    .add("realfeel®", StringType(), True)\
    .add("realfeel shade™", StringType(), True)\
    .add("max uv index", StringType(), True)\
    .add("wind", StringType(), True)\
    .add("wind gusts", StringType(), True)\
    .add("humidity", StringType(), True)\
    .add("indoor humidity", StringType(), True)\
    .add("dew point", StringType(), True)\
    .add("pressure", StringType(), True)\
    .add("cloud cover", StringType(), True)\
    .add("visibility", StringType(), True)\
    .add("cloud ceiling", StringType(), True)

df = spark.readStream\
    .format("kafka")\
    .option("kafka.bootstrap.servers", "localhost:9092")\
    .option("subscribe", "weather-raw")\
    .option("startingOffsets", "earliest")\
    .load()

AnalysisException:  Failed to find data source: kafka. Please deploy the application as per the deployment section of "Structured Streaming + Kafka Integration Guide".        